# Clean and compare generated interpretations

Apply the original cleaning rules and compare three models on the same 35 events.


In [ ]:
from pathlib import Path
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file() and (p / 'data/forensic_atomic.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the forensic_atomic repository.')
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
INPUT_DIR=ROOT/'results/phi4_mini/generation/validated'
OUTPUT_DIR=ROOT/'results/phi4_mini/generation/cleaned'
MODELS=['base','finetuned','deepseek']


## Cleaning rules


In [ ]:
import csv, json, re, math, statistics, random
from collections import Counter,defaultdict
from IPython.display import display

RELATIONS = ['xIntent','xNeed','xAttr','xEffect','xReact','xWant','oEffect','oReact','oWant',
             'isAfter','HasSubEvent','isBefore','HinderedBy','Causes','xReason','isFilledBy']
FIELDS = ['event','crime_category','crime_subcategory','brief_context'] + RELATIONS

import re
import json


_ALLOWED = re.compile(
    r'^['
    r'\x09\x0A\x0D\x20-\x7E'
    r'\u00A0-\u024F'
    r'\u1E00-\u1EFF'
    r'\u2000-\u206F'
    r'\u20A0-\u20CF'
    r']*$',
    re.UNICODE
)

def _is_latin_safe(text: str) -> bool:
    """Returns True if the text contains only allowed Latin/ASCII characters."""
    return bool(_ALLOWED.match(str(text)))

SUBCATEGORY_MAP = {

    "Assault":                    "Assault",
    "Homicide":                   "Homicide",
    "Kidnapping":                 "Kidnapping",
    "Robbery":                    "Robbery",
    "Sexual Offense":             "Sexual Offense",
    "Sexual Offenses":            "Sexual Offense",
    "Stalking":                   "Assault",
    "Extortion":                  "Robbery",
    "Torture":                    "Assault",
    "Threatening Communications": "Assault",
    "Witness Intimidation":       "Obstruction of Justice",


    "Theft":                      "Theft",
    "Burglary":                   "Burglary",
    "Arson":                      "Arson",
    "Vandalism":                  "Vandalism",
    "Trespassing":                "Trespassing",


    "Fraud":                      "Fraud",
    "Insurance Fraud":            "Fraud",
    "Insider Trading":            "Fraud",
    "Counterfeiting":             "Fraud",
    "Money Laundering":           "Money Laundering",
    "Embezzlement":               "Embezzlement",
    "Bribery":                    "Bribery",
    "Tax Evasion":                "Tax Evasion",


    "Hacking":                    "Hacking",
    "Phishing":                   "Phishing",
    "Identity Theft":             "Identity Theft",
    "Online Harassment":          "Online Harassment",
    "Planning a Cyber Attack":    "Hacking",


    "Drug Trafficking":           "Drug Trafficking",
    "Human Trafficking":          "Human Trafficking",
    "Racketeering":               "Racketeering",
    "Weapon Smuggling":           "Weapon Smuggling",
    "Weapon Offense":             "Weapon Smuggling",
    "Weapon Possession":          "Weapon Smuggling",
    "Gang Activity":              "Racketeering",
    "Terrorism":                  "Racketeering",
    "Animal Fighting":            "Racketeering",
    "Dog Fighting":               "Racketeering",
    "Animal Trafficking":         "Human Trafficking",
    "Wildlife Trafficking":       "Human Trafficking",
    "Animal Cruelty":             "Racketeering",
    "Cruelty to Animals":         "Racketeering",

    "Animal Cruelty (implied within trafficking or illegal breeding operations, categorized here as a component of Organized Crime for profit-driven animal exploitation)": "Racketeering",
    "Organized Crimes":           "Racketeering",


    "Perjury":                    "Perjury",
    "Obstruction of Justice":     "Obstruction of Justice",
    "Disorderly Conduct":         "Disorderly Conduct",
}


SUBCATEGORY_TO_CATEGORY = {

    "Assault":                   "Violent Crimes",
    "Homicide":                  "Violent Crimes",
    "Kidnapping":                "Violent Crimes",
    "Robbery":                   "Violent Crimes",
    "Sexual Offense":            "Violent Crimes",

    "Theft":                     "Property Crimes",
    "Burglary":                  "Property Crimes",
    "Arson":                     "Property Crimes",
    "Vandalism":                 "Property Crimes",
    "Trespassing":               "Property Crimes",

    "Fraud":                     "Financial Crimes",
    "Money Laundering":          "Financial Crimes",
    "Embezzlement":              "Financial Crimes",
    "Bribery":                   "Financial Crimes",
    "Tax Evasion":               "Financial Crimes",

    "Hacking":                   "Cyber Crimes",
    "Phishing":                  "Cyber Crimes",
    "Identity Theft":            "Cyber Crimes",
    "Online Harassment":         "Cyber Crimes",

    "Drug Trafficking":          "Organized Crimes",
    "Human Trafficking":         "Organized Crimes",
    "Racketeering":              "Organized Crimes",
    "Weapon Smuggling":          "Organized Crimes",

    "Perjury":                   "Public Order/Justice",
    "Obstruction of Justice":    "Public Order/Justice",
    "Disorderly Conduct":        "Public Order/Justice",
}


def read_csv(path):
    with path.open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))

def write_csv(path,rows,cols):
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=cols,extrasaction='ignore');w.writeheader();w.writerows(rows)
    print(path, 'rows', len(rows))

def norm_event(s):return ' '.join(str(s or '').casefold().split())

def row_is_latin_safe(row):
    for col,val in row.items():
        if val is None or val=='':continue
        val_str=str(val)
        try:
            parsed=json.loads(val_str)
            if isinstance(parsed,list):
                if not all(_is_latin_safe(str(item)) for item in parsed):return False
                continue
        except (ValueError,TypeError):pass
        if not _is_latin_safe(val_str):return False
    return True


## Clean each model


In [ ]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
raw={};clean={};unmapped=[];nonlatin=[];row_changes=[]
for model in MODELS:
    file=INPUT_DIR/f'{model}_post_validator_pre_cleaning.csv'
    if not file.is_file(): raise FileNotFoundError(f'Missing validator input: {file}')
    raw[model]=read_csv(file)
    clean[model]=[]
    for i,row in enumerate(raw[model],1):
        if not row_is_latin_safe(row):
            nonlatin.append({'model':model,'source_row':i,**row})
            continue
        original_sub=str(row.get('crime_subcategory') or '').strip()
        canonical_sub=SUBCATEGORY_MAP.get(original_sub)
        canonical_cat=SUBCATEGORY_TO_CATEGORY.get(canonical_sub)
        out=dict(row)
        out['crime_subcategory']=canonical_sub if canonical_sub is not None else ''
        out['crime_category']=canonical_cat if canonical_cat is not None else ''
        clean[model].append(out)
        if canonical_sub is None:
            unmapped.append({'model':model,'source_row':i,'event':row['event'],
                             'original_category':row.get('crime_category',''),
                             'original_subcategory':original_sub})
        if out['crime_category']!=row.get('crime_category') or out['crime_subcategory']!=original_sub:
            row_changes.append({'model':model,'source_row':i,'event':row['event'],
                                'category_before':row.get('crime_category'),'category_after':out['crime_category'],
                                'subcategory_before':original_sub,'subcategory_after':out['crime_subcategory']})
    write_csv(OUTPUT_DIR/f'{model}_post_original_cleaning.csv',clean[model],FIELDS)
write_csv(OUTPUT_DIR/'excluded_non_latin_rows.csv',nonlatin,['model','source_row']+FIELDS)
write_csv(OUTPUT_DIR/'unmapped_subcategories.csv',unmapped,
          ['model','source_row','event','original_category','original_subcategory'])
write_csv(OUTPUT_DIR/'changed_labels.csv',row_changes,
          ['model','source_row','event','category_before','category_after','subcategory_before','subcategory_after'])


## Paired event statistics


In [ ]:
trace_file=INPUT_DIR/'event_trace.csv'
trace=read_csv(trace_file)
expected={norm_event(row['event']) for row in trace if row['model']=='deepseek'}
if not expected:raise ValueError('No events in the validator report.')
summary=[]
for model in MODELS:
    before=raw[model];after=clean[model]
    covered_before={norm_event(r['event']) for r in before}
    covered_after={norm_event(r['event']) for r in after}
    mapped=[r for r in after if r.get('crime_subcategory') and r.get('crime_category')]
    covered_mapped={norm_event(r['event']) for r in mapped}
    rel_n=Counter()
    for row in after:
        for rel in RELATIONS:
            try:
                v=json.loads(row[rel]);rel_n[rel]+=int(isinstance(v,list) and bool(v))
            except (ValueError,TypeError):pass
    filled_err=sum('___' not in str(r['event']) and bool(json.loads(r['isFilledBy'])) for r in after)
    summary.append({'model':model,'common_events':len(expected),
      'branches_after_validator':len(before),
      'events_with_validated_branches':len(covered_before),
      'branches_after_cleaning':len(after),
      'covered_events_after_cleaning':len(covered_after),
      'branches_with_mapped_taxonomy':len(mapped),
      'events_with_mapped_branch':len(covered_mapped),
      'non_latin_rows_removed':sum(r['model']==model for r in nonlatin),
      'rows_with_unmapped_subcategory':sum(r['model']==model for r in unmapped),
      'rows_with_changed_labels':sum(r['model']==model for r in row_changes),
      'isFilledBy_without_placeholder_diagnostic':filled_err,
      'coverage_pct_after_cleaning':round(100*len(covered_after)/len(expected),2),
      'mapped_coverage_pct':round(100*len(covered_mapped)/len(expected),2),
      'mean_nonempty_relations_per_branch':round(sum(rel_n.values())/len(after),2) if after else 0})
write_csv(OUTPUT_DIR/'01_paired_summary.csv',summary,list(summary[0]))
display(*[r for r in summary])


## Category comparisons


In [ ]:
from itertools import combinations
by_pairs={}
for model in MODELS:
    g=defaultdict(set)
    for row in clean[model]:
        if row['crime_category'] and row['crime_subcategory']:
            g[norm_event(row['event'])].add((row['crime_category'],row['crime_subcategory']))
    by_pairs[model]=g

def jaccard(a,b):
    return round(len(a&b)/len(a|b),4) if (a|b) else ''
paired=[]
for e in sorted(expected):
    values={m:by_pairs[m].get(e,set()) for m in MODELS}
    paired.append({'normalized_event':e,
      **{f'pair_count_{m}':len(values[m]) for m in MODELS},
      **{f'pairs_{m}':' | '.join(sorted('/'.join(x) for x in values[m])) for m in MODELS},
      'jaccard_base_ft':jaccard(values['base'],values['finetuned']),
      'jaccard_base_deepseek':jaccard(values['base'],values['deepseek']),
      'jaccard_ft_deepseek':jaccard(values['finetuned'],values['deepseek'])})
write_csv(OUTPUT_DIR/'02_event_comparison.csv',paired,list(paired[0]))

cats=[];subs=[]
for model in MODELS:
    cats.extend({'model':model,'crime_category':k,'count':v} for k,v in Counter(r['crime_category'] for r in clean[model] if r['crime_category']).most_common())
    subs.extend({'model':model,'crime_subcategory':k,'count':v} for k,v in Counter(r['crime_subcategory'] for r in clean[model] if r['crime_subcategory']).most_common())
write_csv(OUTPUT_DIR/'03_category_distribution.csv',cats,['model','crime_category','count'])
write_csv(OUTPUT_DIR/'04_subcategory_distribution.csv',subs,['model','crime_subcategory','count'])
print('Mean Jaccard (comparable pairs only):')
for col in ['jaccard_base_ft','jaccard_base_deepseek','jaccard_ft_deepseek']:
    vals=[x[col] for x in paired if x[col]!='']
    print(col,round(statistics.mean(vals),3) if vals else 'n/a')


## Summary figures


In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.style.use('default')
    labels=[r['model'] for r in summary]
    fig,ax=plt.subplots(figsize=(8,4))
    x=list(range(len(labels)));v=[r['covered_events_after_cleaning'] for r in summary]
    ax.bar(x,v)
    ax.set_xticks(x,labels);ax.set_ylabel('Events covered after cleaning')
    ax.set_ylim(0,max(len(expected),max(v))+2)
    ax.set_title('Common sample coverage after validation and cleaning')
    plt.tight_layout();plt.savefig(OUTPUT_DIR/'event_coverage.png',dpi=150);plt.show()
except ImportError:
    print('matplotlib unavailable; CSV files were saved.')
print('Reports saved in:',OUTPUT_DIR)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('default')
from collections import Counter


def _pct(n, d):
    return 100 * n / d if d else float('nan')

event_count = len(expected)
percent_rows = []
for model_name in MODELS:
    rows_clean = clean[model_name]
    counts_by_event = Counter(norm_event(r['event']) for r in rows_clean)
    covered_events = {e for e, n in counts_by_event.items() if n > 0}
    mapped_rows = [r for r in rows_clean if r.get('crime_category') and r.get('crime_subcategory')]
    mapped_events = {norm_event(r['event']) for r in mapped_rows}


    without_placeholder = [r for r in rows_clean if '___' not in str(r['event'])]
    n_isfilledby = 0
    for r in without_placeholder:
        try:
            relation_list = json.loads(r.get('isFilledBy') or '[]')
        except (ValueError, TypeError):
            relation_list = []
        n_isfilledby += bool(relation_list)

    changed_count = sum(r['model'] == model_name for r in row_changes)
    percent_rows.append({
        'Model': {'base': 'Phi-4 Base', 'finetuned': 'Phi-4 Fine-tuned',
                    'deepseek': 'DeepSeek'}.get(model_name, model_name),
        'Events covered (%)': _pct(len(covered_events & expected), event_count),
        'Events with >=2 branches (%)': _pct(sum(counts_by_event[e] >= 2 for e in expected), event_count),
        'Events with >=1 mapped branch (%)': _pct(len(mapped_events & expected), event_count),
        'Mapped branches (%)': _pct(len(mapped_rows), len(rows_clean)),
        'Changed labels (%)': _pct(changed_count, len(rows_clean)),
        'isFilledBy without ___ (%) [diagnostic]': _pct(n_isfilledby, len(without_placeholder)),
        'Mean branches per event': len(rows_clean) / event_count,
        'Total branches': len(rows_clean)
    })

pct_df = pd.DataFrame(percent_rows).set_index('Model')
print('Percentage statistics after validation and shared cleaning')
display(pct_df.round(2))
pct_df.to_csv(OUTPUT_DIR / '05_percentage_statistics.csv', encoding='utf-8-sig')


fig, ax = plt.subplots(figsize=(9, 4.5))
pct_df[['Events covered (%)', 'Events with >=2 branches (%)',
        'Events with >=1 mapped branch (%)']].plot.bar(ax=ax, width=0.75)
ax.set(title=f'Coverage across {event_count} common events', ylabel='Events (%)',
       xlabel='', ylim=(0, 108))
ax.tick_params(axis='x', rotation=0)
ax.legend(fontsize=8, loc='lower left')
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_event_percentages.png', dpi=160)
plt.show()


fig, ax = plt.subplots(figsize=(9, 4.5))
pct_df[['Mapped branches (%)', 'Changed labels (%)',
        'isFilledBy without ___ (%) [diagnostic]']].plot.bar(ax=ax, width=0.75)
ax.set(title='Interpretation properties after cleaning',
       ylabel='Relevant branches (%)', xlabel='', ylim=(0, 108))
ax.tick_params(axis='x', rotation=0)
ax.legend(fontsize=8, loc='upper right')
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_branch_percentages.png', dpi=160)
plt.show()


jaccard_rows = []
for field_name, comparison_name in [('jaccard_base_ft', 'Base vs Fine-tuned'),
                    ('jaccard_base_deepseek', 'Base vs DeepSeek'),
                    ('jaccard_ft_deepseek', 'Fine-tuned vs DeepSeek')]:
    values_jaccard = [r[field_name] for r in paired if r[field_name] != '']
    jaccard_rows.append({'Comparison': comparison_name, 'Comparable events': len(values_jaccard),
                    'Mean Jaccard (%)': _pct(sum(values_jaccard), len(values_jaccard)) if values_jaccard else float('nan')})
jacc_df = pd.DataFrame(jaccard_rows).set_index('Comparison')
print('Label similarity; this does not measure forensic quality')
display(jacc_df.round(2))
jacc_df.to_csv(OUTPUT_DIR / '05_jaccard_percentage.csv', encoding='utf-8-sig')

print('Note: isFilledBy without a placeholder only indicates prompt adherence:')
print('      the original validator does not consider it an error.')
print('Note: DeepSeek data come from the processed CSV; base-model branches were recovered from raw output.')
print('Reports and figures saved in:', OUTPUT_DIR)
